# BERT-tiny GooAQ Reranker - AWS Marketplace

Deploys **BERT-tiny GooAQ Reranker** from AWS Marketplace as a SageMaker endpoint inside **your own AWS account**. Your data stays in your account and there are no external API calls.

17.5MB English cross-encoder reranker (4.4M parameters, 2 layers) for RAG and search on ml.t2.medium. MRR@10 0.446 vs 0.351 for random order on 1,000 MS MARCO queries. Apache-2.0, private in your VPC.

## Prerequisites

1. Subscribe to the product in AWS Marketplace.
2. Copy the **model package ARN** shown on the product's launch page for your Region.
3. Run this notebook with a role that has `AmazonSageMakerFullAccess`.

Cost: software fee $0.05/hr plus the SageMaker instance (ml.t2.medium real-time, ml.m5.large batch). Delete the endpoint when done.

In [ ]:
!pip install -qU sagemaker boto3

In [ ]:
import json

import boto3
import sagemaker
from sagemaker import ModelPackage

# Paste the model package ARN from the product's launch page for YOUR Region.
MODEL_PACKAGE_ARN = "<paste-model-package-arn-here>"

INSTANCE_TYPE = "ml.t2.medium"  # smallest real-time type, enough for this model
ENDPOINT_NAME = "reranker-bert-tiny-gooaq-bce"

session = sagemaker.Session()
role = sagemaker.get_execution_role()
print("region:", session.boto_region_name)

## 2. Deploy a real-time endpoint

Creating the endpoint takes a few minutes. It bills per hour while it exists, so do not skip section 5.

In [ ]:
model = ModelPackage(
    role=role,
    model_package_arn=MODEL_PACKAGE_ARN,
    sagemaker_session=session,
)

predictor = model.deploy(
    initial_instance_count=1,
    instance_type=INSTANCE_TYPE,
    endpoint_name=ENDPOINT_NAME,
)
print("endpoint ready:", ENDPOINT_NAME)

## 3. Rank passages

The endpoint accepts `{"query": "...", "passages": ["...", "..."], "top_k": 2}` and returns `scores` (raw logits), `relevance` (0-1), `ranking` (passage indexes, best first) and `top_k`.

In [ ]:
runtime = boto3.client("sagemaker-runtime")

PAYLOAD = {'query': 'how do I reset my password', 'passages': ['Click Forgot password on the sign-in page.', 'Our office is open from 9 to 5.'], 'top_k': 2}

response = runtime.invoke_endpoint(
    EndpointName=ENDPOINT_NAME,
    ContentType="application/json",
    Body=json.dumps(PAYLOAD),
)
result = json.loads(response["Body"].read())
print(json.dumps(result)[:600])

ranked = [PAYLOAD["passages"][i] for i in result["ranking"]]
for i in result["ranking"]:
    print(f"{result['scores'][i]:.3f}  {PAYLOAD['passages'][i]}")

## 4. Batch transform for offline workloads

Batch transform avoids paying for an always-on endpoint (ml.m5.large). The input file is sent as ONE request, so put a single JSON request object (same shape as above) in the file.

In [ ]:
# transformer = model.transformer(
#     instance_count=1,
#     instance_type="ml.m5.large",
#     output_path=f"s3://{session.default_bucket()}/reranker-bert-tiny-gooaq-bce/",
#     strategy="SingleRecord",
# )
# transformer.transform(
#     data=f"s3://{session.default_bucket()}/reranker-bert-tiny-gooaq-bce-input/request.jsonl",
#     content_type="application/json",
# )
# transformer.wait()

## 5. Clean up

Delete the endpoint when you are done. It bills for as long as it is running.

In [ ]:
predictor.delete_model()
predictor.delete_endpoint()
print("deleted:", ENDPOINT_NAME)